In [10]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objs as go
from dotenv import load_dotenv

from datetime import datetime
import yfinance as yf

import requests
import csv
import math

import warnings


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")
pd.options.mode.copy_on_write = True

In [11]:
load_dotenv()

API_KEY = os.getenv("alpha_vantage_api_key")
# local_inv_directry = os.getenv("local_inv_directry")

alpha_vantage_api_key = API_KEY

In [12]:
# Change working directory to D:\Investment
os.chdir('D:\Investment')

# Functional Table
## Forex Tables

In [13]:
# This forex session tables will apply to all the forex calculation below
# CADUSD source: https://www.investing.com/currencies/cad-usd-historical-data
# Methdology: source table is monthly table CSV downloaded from inveseting.com, 
# we will aggregate it to the yearly level by extracting the last month(DEC) forex data


fx_CADUSD_df = pd.read_csv("CAD_USD Historical Data.csv")

fx_CADUSD_df['Date'] = pd.to_datetime(fx_CADUSD_df['Date'])
fx_CADUSD_df['Year'] = fx_CADUSD_df['Date'].dt.year
fx_CADUSD_yrly_df = fx_CADUSD_df.groupby('Year')['Price'].mean().to_frame(name='CADUSD')
fx_CADUSD_yrly_df = fx_CADUSD_yrly_df.sort_index(ascending=False)
fx_CADUSD_yrly_df['Year'] = fx_CADUSD_yrly_df.index # create the Year column

In [14]:
fx_CADUSD_yrly_df

,CADUSD,Year
Year,,
2026,0.737100,2026
2025,0.715667,2025
2024,0.728400,2024
2023,0.741775,2023
2022,0.766983,2022
2021,0.797967,2021
2020,0.746617,2020
2019,0.755675,2019
2018,0.769458,2018


# SP500 Anuual Hist Return

In [15]:
# SP 500 Daily quote section
# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=SPY&apikey={alpha_vantage_api_key}&outputsize=full'
r = requests.get(url)
data = r.json()

for key, value in data.items():
    if key == 'Time Series (Daily)':


        selected_cols = [
            '4. close'
        ]

        sp500_Daily_stock_df = pd.DataFrame(value).transpose()[selected_cols] # tranpose the dataframe and sub select selected cols

        # Rename columns
        sp500_Daily_stock_df.rename(
            columns={
                '4. close': 'stock_price'
                }
            ,inplace=True
            )
        
        sp500_Daily_stock_df["stock_price"] = sp500_Daily_stock_df["stock_price"].astype(str).apply(lambda x: float(x))
        sp500_Daily_stock_df["stock_price"] = sp500_Daily_stock_df["stock_price"].round(2)
        sp500_Daily_stock_df.index = pd.to_datetime(sp500_Daily_stock_df.index)


sp500_hist_annual = sp500_Daily_stock_df.resample('Y').last()  # Get the last entry of each year
sp500_hist_annual.index = sp500_hist_annual.index.year
sp500_hist_annual['Year']= sp500_hist_annual.index

# Calculate the percentage change year over year
sp500_hist_annual['Annual Return'] = sp500_hist_annual['stock_price'].pct_change() * 100

# sort by years
sp500_hist_annual = sp500_hist_annual.sort_index(ascending=False)

# Drop NaN values
sp500_hist_annual = sp500_hist_annual.dropna(subset=['Annual Return'])

# Data Source Gathering: investment return summary

## Brokerage: Questrade

In [16]:
# Questrade transaction history only back up for 5 years, so the total transactions data is made of historical data plus current year data

# this is the current year file
inv_activity_source_path = 'questrade_inv_activity.xlsx'
inv_activity_df = pd.read_excel(inv_activity_source_path)

# this is the histrical file, the historical file will contain data from year2019 to the year prior to CURRENT YEAR
inv_activity_df_origin = pd.read_excel('questrade_Inv_activity_origin.xlsx')

In [17]:
# Convert 'Transaction Date' column to datetime if it's not already
inv_activity_df_origin['Transaction Date'] = pd.to_datetime(inv_activity_df_origin['Transaction Date'])

# For the historical file, which is the orgin excel, filter out rows where the year in 'Transaction Date' less than CURRENT YEAR
inv_activity_df_orgin_filtered = inv_activity_df_origin[inv_activity_df_origin['Transaction Date'].dt.year < datetime.today().year]
inv_activity_df['Transaction Date'] = pd.to_datetime(inv_activity_df['Transaction Date'])



inv_activity_df = pd.concat(
    [
        inv_activity_df_orgin_filtered
        ,inv_activity_df
        ]
        ,ignore_index=True
        )

In [18]:
# export the combined historical plus current year data to questrade_Inv_activity_origin.xlsx file
# inv_activity_df.to_excel('questrade_Inv_activity_origin.xlsx')

In [19]:
# Note:
# Need to execute the code in github to allow API_KEY being digested


# since the transcation history only include the latest 5 yrs data, we will create a dictonary to store the info each year 
# 1. capital gain/loss
# 2. open securities
# 3. open securities market value at the year end month(12/31)
# 4. total dividends of the year
# 5. total deposits of the year_totals
# 6. total withdrawl of the year

# we start rolling the year 2019, as it has the most complete transactions info

# finished the aggregate level inv_yrly_merged
# finished the stock holding cost consolidated_holding_cost_df

# will need the new logic to allow replace current year transactions to the original dataframe

In [20]:
# Parameter's control Pannel which update the latest year, and add info to the dictionary

inv_activity_df['Transaction Date'] = pd.to_datetime(inv_activity_df['Transaction Date'])
inv_activity_df['Year'] = inv_activity_df['Transaction Date'].dt.year

filter_1 = (inv_activity_df['Year'] >= 2019)


inv_activity_df = inv_activity_df[
    filter_1
    ]

inv_activity_df.rename(
    columns={
    'Transaction Date': 'Transaction_Date'
    ,'Settlement Date': 'Settlement_Date'
    ,'Net Amount': 'Net_Amount'
    }
    ,inplace=True
    )


## Enrichment on DEPOSIT, WITHDRAWAL, DIVIDEND

In [21]:
deposit_withdrawl_divdend_df = inv_activity_df

# # FOREX section - FOREX df to exchange CAD dividend to USD dividend
# # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
# url = f'https://www.alphavantage.co/query?function=FX_DAILY&from_symbol=CAD&to_symbol=USD&outputsize=full&apikey={alpha_vantage_api_key}'
# r = requests.get(url)
# data = r.json()

# for key ,value in data.items():

#     if key == 'Time Series FX (Daily)':
#         fx_CADUSD_df = pd.DataFrame(value).transpose()[['4. close']]
#         # change str to num
#         fx_CADUSD_df['4. close'] = pd.to_numeric(fx_CADUSD_df['4. close'], errors='coerce')
        
#         # Rename the column
#         fx_CADUSD_df.rename(
#             columns={'4. close': 'CADUSD'}
#             ,inplace=True
#             )

#         fx_CADUSD_df.index = pd.to_datetime(fx_CADUSD_df.index)
#         fx_CADUSD_yrly_df = fx_CADUSD_df[['CADUSD']].resample('Y').mean()
#         fx_CADUSD_yrly_df.index = fx_CADUSD_yrly_df.index.year
#         fx_CADUSD_yrly_df = fx_CADUSD_yrly_df.sort_index(ascending=False)




# Define action categories and their corresponding actions
action_categories_dict = {
    'Deposits': [
        'CON'
        ,'FCH'
        ,'DEP'
        ], #CON is the contribution in TFSA resigered account, DEP is the deposit in cash margin account, FCH is journaling cost
    'Withdrawals': ['WDR'],
    'Dividends': ['DIV', 'Dividends']
}

# Only update rows with Currency == 'CAD' and Activity Type == 'Dividends'
action_categories_condition = (deposit_withdrawl_divdend_df['Currency'] == 'CAD') & (deposit_withdrawl_divdend_df['Activity Type'] == 'Dividends')

# Initialize a dictionary to hold the yearly totals for each category
yearly_deposit_withdrawl_dividend_dict = {}

# Loop through each category to filter and aggregate
for category, actions in action_categories_dict.items():
    # Filter the DataFrame for the current category's actions
    filtered_df = deposit_withdrawl_divdend_df[(deposit_withdrawl_divdend_df['Action'].isin(actions)) 
                                               | (deposit_withdrawl_divdend_df['Activity Type'].isin(actions))]
    
    # Perform the conversion using the forex rate from fx_CADUSD_yrly_df
    filtered_df.loc[action_categories_condition, 'Net_Amount'] = filtered_df.loc[action_categories_condition].apply(
        lambda row: row['Net_Amount'] * fx_CADUSD_yrly_df.loc[row['Year'], 'CADUSD'],
        axis=1
    )   
    # Sum 'Net_Amount' for other categories
    yearly_deposit_withdrawl_dividend_dict[category] = filtered_df.groupby(
        [
            'Year'
            ]
        )['Net_Amount'].sum()
        

# Note:
# 1. dividend currency can in US or CAD or any other depends on the stock 
inv_yrly_base = pd.DataFrame(yearly_deposit_withdrawl_dividend_dict)
inv_yrly_base.fillna(0, inplace=True)

# Mannually change the 2019 deposits to be 6000
inv_yrly_base.at[2019, 'Deposits'] = 6000

In [22]:
inv_yrly_base

,Deposits,Withdrawals,Dividends
Year,,,
2019,6000.00,0.0,15.060000
2020,5825.00,0.0,9.420000
2021,10200.00,0.0,65.330000
2022,2343.00,-14000.0,55.500000
2023,10000.00,0.0,69.320000
2024,21843.89,0.0,183.210000
2025,140000.00,0.0,2328.995883
2026,-11.24,0.0,785.725988


In [23]:
filter_2 = (inv_activity_df['Activity Type'].isin([
    'Trades'
    ]))

filter_3 = (~inv_activity_df['Symbol'].isin(['DLR.TO' # to exclude all the Norbert's Gambit transactions, this is buy curreny step
                                             ,'G036247'
                                             ,'H038778'
                                             ,'DLR'
                                             ,'IVV' # to exclude trades are not traceable
                                             ,'VYM'
                                             ,'VFH'
                                             ,'KWEB'
                                             ,'BYND'
                                             ])) 


trades_df_questrade = inv_activity_df[
    filter_2
    & filter_3
    ]

In [24]:
# if no trade records in current year, creating sythentic trades record to document the previous year's open transactions
if datetime.today().year not in sorted(trades_df_questrade['Year'].unique()):
    print('No current year transactions in Questrade, need to create sythetic buy & sell transactions')

    trades_df_questrade = trades_df_questrade.sort_values(by='Transaction_Date').reset_index(drop=True)

    # get the last row and duplicate sythetic a buy & a sythetic sell transactions
    last_row = trades_df_questrade.iloc[-1]

    # clean sythetic dataframes
    last_row_buy_df = pd.DataFrame([last_row])
    last_row_buy_df['Action'] = 'Buy'
    last_row_buy_df['Transaction_Date'] = pd.to_datetime(f"{datetime.now().year}-01-01 00:00:00")
    last_row_buy_df['Settlement_Date'] = pd.to_datetime(f"{datetime.now().year}-01-01 00:00:00")
    last_row_buy_df['Year'] = datetime.now().year
    last_row_buy_df['Quantity'] = abs(last_row['Quantity'])
    last_row_buy_df['Net_Amount'] = (abs(last_row['Quantity']) * last_row['Price']) * -1


    last_row_sell_df = pd.DataFrame([last_row])
    last_row_sell_df['Action'] = 'Sell'
    last_row_sell_df['Transaction_Date'] = pd.to_datetime(f"{datetime.now().year}-01-02 00:00:00") # 1 day delay for the .idxmax()
    last_row_sell_df['Settlement_Date'] = pd.to_datetime(f"{datetime.now().year}-01-02 00:00:00")
    last_row_sell_df['Year'] = datetime.now().year
    last_row_sell_df['Quantity'] = abs(last_row['Quantity']) * -1
    last_row_sell_df['Net_Amount'] = (abs(last_row['Quantity']) * last_row['Price'])


    trades_df_questrade = pd.concat(
        [
            trades_df_questrade
            ,last_row_buy_df
            ,last_row_sell_df
            ]
            ,ignore_index=True
            )
else:
    print('No need to create synthetic data')
    pass

No need to create synthetic data


## Enrichment on transactions

In [25]:
# convert CAD trades to USD in questrade trades df

trades_df_questrade_filter_1 = (trades_df_questrade['Currency'] == 'CAD')

# Perform the conversion using the forex rate from fx_CADUSD_yrly_df
# forex on 'Net_Amount'
trades_df_questrade.loc[trades_df_questrade_filter_1, 'Net_Amount'] = trades_df_questrade.loc[trades_df_questrade_filter_1].apply(
    lambda row: row['Net_Amount'] * fx_CADUSD_yrly_df.loc[row['Year'], 'CADUSD'],
    axis=1
)   

# forex on Price
trades_df_questrade.loc[trades_df_questrade_filter_1, 'Price'] = trades_df_questrade.loc[trades_df_questrade_filter_1].apply(
    lambda row: row['Price'] * fx_CADUSD_yrly_df.loc[row['Year'], 'CADUSD'],
    axis=1
)   

In [26]:
# Initialize open_buy_df_previous as an empty DataFrame
open_buy_df_previous = pd.DataFrame()

# sort the year from the smallest to the largest to match open stock sequence
for i in sorted(trades_df_questrade['Year'].unique()):

    # create empty open_stock_mkt_value_list to store yearly open stock values
    open_stock_mkt_value_list = []
    yearly_trades_df = trades_df_questrade[trades_df_questrade['Year'] == i]
    yearly_closed_gain_loss_amount = yearly_trades_df['Net_Amount'].sum()
    # first assign net_amount to the inv_yrly_base
    inv_yrly_base.loc[inv_yrly_base.index == i, 'Closed_gain/loss'] = yearly_closed_gain_loss_amount


    # open df process
    # if not open_buy_df_previous.empty:
    yearly_trades_df = pd.concat([yearly_trades_df, open_buy_df_previous], ignore_index=True)

    buy_filter = (yearly_trades_df['Action'] == 'Buy')
    sell_filter = (yearly_trades_df['Action'] == 'Sell')

    buy_df = yearly_trades_df[buy_filter]
    sell_df = yearly_trades_df[sell_filter]

    buy_ticker_grouped = buy_df.groupby('Symbol')['Quantity'].sum()
    sell_ticker_grouped = sell_df.groupby('Symbol')['Quantity'].sum()

    # Fill NaN values in sell_grouped with 0 (indicating no sales for those symbols)
    sell_ticker_grouped = sell_ticker_grouped.reindex(buy_ticker_grouped.index, fill_value=0)
    # use '+' sign due to sell_groups contains negative quantity
    open_ticker_grouped = buy_ticker_grouped + sell_ticker_grouped
    open_ticker_grouped_df = open_ticker_grouped[open_ticker_grouped>0].reset_index()


    # Merging the open stock symbols with their details from the buy_df to create the open_stock_df
    open_buy_df = pd.merge(
        open_ticker_grouped_df
        ,buy_df
        ,on='Symbol'
        ,how='left'
        )
    
    open_buy_df = open_buy_df.drop_duplicates(
        subset=[
            'Symbol'
            ,'Quantity_x'
            ]
            ,keep='first'
            )


    # open_buy_df cleaning process
    selected_cols = [
    'Transaction_Date'
    ,'Settlement_Date'
    ,'Action'
    ,'Symbol'
    ,'Description'
    ,'Quantity_x'
    ,'Price'
    ,'Gross Amount'
    ,'Commission'
    ,'Net_Amount'
    ,'Currency'
    ,'Account #'
    ,'Activity Type'
    ,'Account Type'
    ,'Year'
    ]

    open_buy_df = open_buy_df[selected_cols]

    open_buy_df.rename(columns={
        'Quantity_x': 'Quantity'
        }
        ,inplace=True
        )


    buy_shares_qty = abs(buy_ticker_grouped.sum())
    sell_shares_qty = abs(sell_ticker_grouped.sum())
    open_shares_qty = abs(buy_shares_qty - sell_shares_qty)


    # Save the current open_buy_df to be used in the next year's iteration
    open_buy_df_previous = open_buy_df.copy()

    print(i, open_shares_qty, open_buy_df['Symbol'].unique())
    # iterate the open stock list
    for symbol in open_buy_df['Symbol'].unique():

        ########################################################################
        # ATTENTION!!!, here the Canadian Open stock also need to forex to USD #
        # BUT here, we need to mannual convert the CAD security to USD         #
        ########################################################################

        if symbol in [
            'CMR.TO'
            ]:

            # STOCK SPLIT FACTOR section
            url = f'https://www.alphavantage.co/query?function=SPLITS&symbol={symbol}&apikey={alpha_vantage_api_key}'
            r = requests.get(url)
            data = r.json()

            for key, value in data.items():
                if key == 'data':
                    if len(value) > 0:
                        stock_split_record_df = pd.DataFrame(value)
                        stock_split_record_df['split_factor'] = pd.to_numeric(stock_split_record_df['split_factor'], errors='coerce') # change split_factor series to numeric data
                        stock_split_record_df['effective_date'] = pd.to_datetime(stock_split_record_df['effective_date'])
                    else:
                        # Use pandas index assign instead of direct assign value
                        stock_split_record_df = pd.DataFrame({
                        'split_factor': [1],
                        'effective_date': [datetime.today()]
                    })

            
            
            # Daily quote section
            # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
            url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={alpha_vantage_api_key}&outputsize=full'
            r = requests.get(url)
            data = r.json()

            for key, value in data.items():
                if key == 'Time Series (Daily)':


                    selected_cols = [
                        '4. close'
                    ]

                    Daily_stock_df = pd.DataFrame(value).transpose()[selected_cols] # tranpose the dataframe and sub select selected cols

                    # Rename columns
                    Daily_stock_df.rename(
                        columns={
                            '4. close': 'stock_price'
                            }
                        ,inplace=True
                        )
                    
                    Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].astype(str).apply(lambda x: float(x))
                    Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].round(2)
                    Daily_stock_df.index = pd.to_datetime(Daily_stock_df.index)


            for date_i in Daily_stock_df.index.date:
                for date_j in stock_split_record_df['effective_date'].dt.date:
                    if date_i == date_j:

                        # stock price to divided the split factor
                        Daily_stock_df.loc[Daily_stock_df.index.date < date_j, 'stock_price'] /= (stock_split_record_df['split_factor'][stock_split_record_df['effective_date'].dt.date == date_j].values[0])


            # Group by year and get the last entry for each year, for the current year, will get the latest date quote
            latest_Daily_stock_df = Daily_stock_df.resample('Y').last()
            
            # merge to get the forex to USD
            latest_Daily_stock_df['Year'] = latest_Daily_stock_df.index.year
            fx_CADUSD_yrly_df = fx_CADUSD_yrly_df.reset_index(drop=True)
            # # Reset index of fx_CADUSD_yrly_df to make 'Year' a column
            # fx_CADUSD_yrly_df = fx_CADUSD_yrly_df.reset_index()
            # fx_CADUSD_yrly_df = fx_CADUSD_yrly_df.rename(columns={'index': 'Year'})  # in case it becomes 'index'

            # merged_latest_Daily_stock_df = latest_Daily_stock_df.merge(
            #     fx_CADUSD_yrly_df
            #     ,on='Year'
            #     ,how='left'
            # )

            merged_latest_Daily_stock_df = latest_Daily_stock_df.merge(
                    fx_CADUSD_yrly_df
                    ,on='Year'
                    ,how='left'
                )

            merged_latest_Daily_stock_df['stock_price'] = merged_latest_Daily_stock_df['stock_price'] * merged_latest_Daily_stock_df['CADUSD']



            open_stock_price = merged_latest_Daily_stock_df[merged_latest_Daily_stock_df['Year'] == i].values[0][0]
            open_stock_qty = open_buy_df[open_buy_df['Symbol'] == symbol]['Quantity'].values.sum()


            open_stock_mkt_value = (open_stock_qty * open_stock_price).sum()
            open_stock_mkt_value_list.append(open_stock_mkt_value)
            print(symbol, open_stock_qty, open_stock_price, open_stock_mkt_value)


        else:

            # STOCK SPLIT FACTOR section
            url = f'https://www.alphavantage.co/query?function=SPLITS&symbol={symbol}&apikey={alpha_vantage_api_key}'
            r = requests.get(url)
            data = r.json()

            for key, value in data.items():
                if key == 'data':
                    if len(value) > 0:
                        stock_split_record_df = pd.DataFrame(value)
                        stock_split_record_df['split_factor'] = pd.to_numeric(stock_split_record_df['split_factor'], errors='coerce') # change split_factor series to numeric data
                        stock_split_record_df['effective_date'] = pd.to_datetime(stock_split_record_df['effective_date'])
                    else:
                        # Use pandas index assign instead of direct assign value
                        stock_split_record_df = pd.DataFrame({
                        'split_factor': [1],
                        'effective_date': [datetime.today()]
                    })
                    
            
            # Daily quote section
            # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
            url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={alpha_vantage_api_key}&outputsize=full'
            r = requests.get(url)
            data = r.json()

            for key, value in data.items():
                if key == 'Time Series (Daily)':


                    selected_cols = [
                        '4. close'
                    ]

                    Daily_stock_df = pd.DataFrame(value).transpose()[selected_cols] # tranpose the dataframe and sub select selected cols

                    # Rename columns
                    Daily_stock_df.rename(
                        columns={
                            '4. close': 'stock_price'
                            }
                        ,inplace=True
                        )
                    
                    Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].astype(str).apply(lambda x: float(x))
                    Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].round(2)
                    Daily_stock_df.index = pd.to_datetime(Daily_stock_df.index)


            for date_i in Daily_stock_df.index.date:
                for date_j in stock_split_record_df['effective_date'].dt.date:
                    if date_i == date_j:

                        # stock price to divided the split factor
                        Daily_stock_df.loc[Daily_stock_df.index.date < date_j, 'stock_price'] /= (stock_split_record_df['split_factor'][stock_split_record_df['effective_date'].dt.date == date_j].values[0])


            # Group by year and get the last entry for each year, for the current year, will get the latest date quote
            latest_Daily_stock_df = Daily_stock_df.resample('Y').last()

            open_stock_price = latest_Daily_stock_df[latest_Daily_stock_df.index.year == i].values[0][0]
            open_stock_qty = open_buy_df[open_buy_df['Symbol'] == symbol]['Quantity'].values.sum()


            open_stock_mkt_value = (open_stock_qty * open_stock_price).sum()
            open_stock_mkt_value_list.append(open_stock_mkt_value)
            print(symbol, open_stock_qty, open_stock_price, open_stock_mkt_value)


    print(round(sum(open_stock_mkt_value_list), 2))
    print()
    inv_yrly_base.loc[inv_yrly_base.index == i, 'Open_mkt_value'] = round(sum(open_stock_mkt_value_list), 2)




2019 41.0 ['BABA' 'DDOG' 'ORCL']
BABA 20.0 212.1 4242.0
DDOG 9.0 37.78 340.02
ORCL 12.0 52.98 635.76
5217.78

2020 72.0 ['AAL' 'AMZN' 'DDOG']
AAL 20.0 15.77 315.4
AMZN 2.0 162.8465 325.693
DDOG 50.0 98.44 4922.0
5563.09

2021 50.0 ['FDX' 'META' 'TSM']
FDX 10.0 258.64 2586.3999999999996
META 20.0 336.35 6727.0
TSM 20.0 120.31 2406.2
11719.6

2022 50.0 ['FDX' 'META' 'TSM']
FDX 10.0 173.2 1732.0
META 20.0 120.34 2406.8
TSM 20.0 74.49 1489.8
5628.6

2023 40.0 ['TSM']
TSM 40.0 104.0 4160.0
4160.0

2024 109.0 ['GOOG' 'TGT' 'TSM']
GOOG 67.0 190.44 12759.48
TGT 22.0 135.18 2973.96
TSM 20.0 197.49 3949.8
19683.24

2025 3197.0 ['ACN' 'CL' 'CMR.TO' 'GOOG' 'NVDA' 'SGOV' 'TGT' 'TSM']
ACN 7.0 268.3 1878.1000000000001
CL 112.0 79.02 8850.24
CMR.TO 2796.0 35.790490000000005 100070.21004000002
GOOG 10.0 313.8 3138.0
NVDA 28.0 186.5 5222.0
SGOV 217.0 100.38 21782.46
TGT 22.0 97.75 2150.5
TSM 5.0 303.89 1519.4499999999998
144610.96

2026 2119.0 ['ACN' 'CMR.TO' 'GOOG' 'INTU' 'MSFT' 'NVDA' 'SKWD' 'TGT' 'TS

In [27]:
# hard code adding 1663 USD to year 2026, because missing 1663 USD when did DLR.TO to DLR.USD(CAD to USD forex)
# inv_yrly_base.at[2026, 'Closed_gain/loss'] += 1663

In [28]:
inv_yrly_base

,Deposits,Withdrawals,Dividends,Closed_gain/loss,Open_mkt_value
Year,,,,,
2019,6000.00,0.0,15.060000,-4668.270000,5217.78
2020,5825.00,0.0,9.420000,-3162.900000,5563.09
2021,10200.00,0.0,65.330000,1641.390000,11719.60
2022,2343.00,-14000.0,55.500000,118.220000,5628.60
2023,10000.00,0.0,69.320000,8470.100000,4160.00
2024,21843.89,0.0,183.210000,-9934.380000,19683.24
2025,140000.00,0.0,2328.995883,-120970.430280,144610.96
2026,-11.24,0.0,785.725988,7813.010466,142381.91


## Brokerage: MOOMOO

In [29]:
# Moomoo transaction history, the spreadsheet contains YTD transactions
# Moomoo transaction can only export 1 Yr from the web, so the full transactions will combine all years historical together
inv_activity_source_path_2024 = 'individual_activities_2024.csv'
inv_activity_source_path_2025 = 'individual_activities_2025.csv'
inv_activity_source_path_2026 = 'individual_activities_2026.csv'

inv_activity_df_moomoo_2024 = pd.read_csv(inv_activity_source_path_2024)
inv_activity_df_moomoo_2025 = pd.read_csv(inv_activity_source_path_2025)
inv_activity_df_moomoo_2026 = pd.read_csv(inv_activity_source_path_2026)

inv_activity_df_moomoo = pd.concat([
    inv_activity_df_moomoo_2024
    ,inv_activity_df_moomoo_2025
    ,inv_activity_df_moomoo_2026
])

In [30]:
selected_cols = [
    # DATE DIMENSION
    'ReportDate'

    ,'ActivityCode'
    ,'Buy/Sell'
    ,'TradeQuantity'
    ,'TradeGross'
    ,'Amount'

    ,'CurrencyPrimary'
    ,'AssetClass'
    ,'Symbol'
    ,'TransactionID'
]

inv_activity_df_moomoo = inv_activity_df_moomoo[selected_cols]

# Assuming 'df' is the DataFrame
inv_activity_df_moomoo['ReportDate'] = pd.to_datetime(inv_activity_df_moomoo['ReportDate'], format='%Y%m%d')
# Assuming 'df' is the DataFrame
inv_activity_df_moomoo['Year'] = inv_activity_df_moomoo['ReportDate'].dt.year

In [31]:
filter_1 = (inv_activity_df_moomoo['CurrencyPrimary'] == 'CAD')
filter_2 = (inv_activity_df_moomoo['ActivityCode'] == 'DEP')
filter_3 = (inv_activity_df_moomoo['ActivityCode'] == 'DIV')
filter_4 = (inv_activity_df_moomoo['ActivityCode'] == 'WITH')

filter_5 = (inv_activity_df_moomoo['AssetClass'].isnull())

In [32]:
inv_activity_df_moomoo = inv_activity_df_moomoo[
    filter_1 
    & filter_2 
    | filter_3
    | filter_4
    ].sort_values(by='ReportDate', ascending=True) \
    .drop_duplicates(
        [
            'TransactionID'
        ]
    )

## Enrichment on DEPOSIT, WITHDRAWAL, DIVIDEND

In [33]:
deposit_withdrawl_divdend_df_moomoo = inv_activity_df_moomoo

# There are 3 actions introduced in the data
action_dict = {
    'Deposits':'DEP'
    ,'Withdrawls':'WITH'
    ,'Dividends':'DIV'
    # ,'Reffered_reward':'FCH'

}

# Define action categories and their corresponding actions
action_categories_dict = {
    'Deposits': ['DEP'],
    'Withdrawals': ['WITH'],
    'Dividends': ['DIV']
}

# Initialize a dictionary to hold the yearly totals for each category
yearly_deposit_withdrawl_dividend_dict_moomoo = {}

# Loop through each category to filter and aggregate
for category, actions in action_categories_dict.items():
    # Filter the DataFrame for the current category's actions
    filtered_df_moomoo = deposit_withdrawl_divdend_df_moomoo[deposit_withdrawl_divdend_df_moomoo['ActivityCode'].isin(actions)]
    
    # Sum 'Net_Amount' for other categories
    yearly_deposit_withdrawl_dividend_dict_moomoo[category] = filtered_df_moomoo.groupby(
        [
            'Year'
            ]
        )['Amount'].sum()
        

# Note:
# 1. dividend currency can in US or CAD or any other depends on the stock 
inv_yrly_base_moomoo = pd.DataFrame(yearly_deposit_withdrawl_dividend_dict_moomoo)
inv_yrly_base_moomoo.fillna(0, inplace=True)

In [34]:
# in CAD
inv_yrly_base_moomoo

,Deposits,Withdrawals,Dividends
Year,,,
2024,50802.884448,0.00,134.615655
2025,27500.000000,-12055.89,974.482836
2026,0.000000,-10000.00,0.000000


## Enrichment on transactions

In [35]:
inv_activity_source_path_2024 = 'individual_activities_2024.csv'
inv_activity_source_path_2025 = 'individual_activities_2025.csv'
inv_activity_source_path_2026 = 'individual_activities_2026.csv'

inv_activity_df_moomoo_2024 = pd.read_csv(inv_activity_source_path_2024)
inv_activity_df_moomoo_2025 = pd.read_csv(inv_activity_source_path_2025)
inv_activity_df_moomoo_2026 = pd.read_csv(inv_activity_source_path_2026)

inv_activity_df_moomoo = pd.concat([
    inv_activity_df_moomoo_2024
    ,inv_activity_df_moomoo_2025
    ,inv_activity_df_moomoo_2026
])

In [36]:
selected_cols = [
    # DATE DIMENSION
    'ReportDate'

    ,'ActivityCode'
    ,'Buy/Sell'
    ,'TradeQuantity'
    ,'TradeGross'
    ,'Amount'

    ,'CurrencyPrimary'
    ,'AssetClass'
    ,'Symbol'
]

inv_activity_df_moomoo = inv_activity_df_moomoo[selected_cols]

# Assuming 'df' is the DataFrame
inv_activity_df_moomoo['ReportDate'] = pd.to_datetime(inv_activity_df_moomoo['ReportDate'], format='%Y%m%d')
# Assuming 'df' is the DataFrame
inv_activity_df_moomoo['Year'] = inv_activity_df_moomoo['ReportDate'].dt.year

In [37]:
filter_2 = (inv_activity_df_moomoo['ActivityCode'].isin([
    'FOREX'
    ,'BUY'
    ,'SELL'
    ]))

filter_3 = (inv_activity_df_moomoo['AssetClass'].isin([
    'CASH'
    ,'STK'
    ]))

filter_4 = (inv_activity_df_moomoo['CurrencyPrimary'].isin([
    'USD'
    ]))

trades_df_moomoo = inv_activity_df_moomoo[
    filter_2
    & filter_3
    & filter_4
    ]

In [38]:
# if no trade records in current year, creating sythentic trades record to document the previous year's open transactions
if datetime.today().year not in sorted(trades_df_moomoo['Year'].unique()):
    print('No current year transactions in Moomoo, need to create sythetic buy & sell transactions')

    trades_df_moomoo = trades_df_moomoo.sort_values(by='ReportDate').reset_index(drop=True)

    # get the last row and duplicate sythetic a buy & a sythetic sell transactions
    last_row = trades_df_moomoo.iloc[-1]

    # clean sythetic dataframes
    last_row_buy_df = pd.DataFrame([last_row])
    last_row_buy_df['ActivityCode'] = 'Buy'
    last_row_buy_df['Buy/Sell'] = 'Buy'
    last_row_buy_df['ReportDate'] = f"{datetime.now().year}-01-01"
    last_row_buy_df['Year'] = datetime.now().year
    last_row_buy_df['TradeQuantity'] = abs(last_row['TradeQuantity'])
    last_row_buy_df['Amount'] = (abs(last_row['Amount'])) * -1


    last_row_sell_df = pd.DataFrame([last_row])
    last_row_sell_df['ActivityCode'] = 'Sell'
    last_row_sell_df['Buy/Sell'] = 'Sell'
    last_row_sell_df['ReportDate'] = f"{datetime.now().year}-01-02" # 1 day delay for the .idxmax()
    last_row_sell_df['Year'] = datetime.now().year
    last_row_sell_df['TradeQuantity'] = abs(last_row['TradeQuantity']) * -1
    last_row_sell_df['Amount'] = (abs(last_row['Amount']))


    trades_df_moomoo = pd.concat(
        [
            trades_df_moomoo
            ,last_row_buy_df
            ,last_row_sell_df
            ]
            ,ignore_index=True
            )
else:
    print('No need to create synthetic data')
    pass

No need to create synthetic data


In [39]:
# Initialize open_buy_df_previous as an empty DataFrame
open_buy_df_moomoo_previous = pd.DataFrame()


# sort the year from the smallest to the largest to match open stock sequence
for i in sorted(trades_df_moomoo['Year'].unique()):

    # create empty open_stock_mkt_value_list to store yearly open stock values
    open_stock_mkt_value_moomoo_list = []
    yearly_trades_df_moomoo = trades_df_moomoo[trades_df_moomoo['Year'] == i]
    yearly_closed_gain_loss_amount_moomoo = yearly_trades_df_moomoo['Amount'].sum()
    # first assign net_amount to the inv_yrly_base
    inv_yrly_base_moomoo.loc[inv_yrly_base_moomoo.index == i, 'Closed_gain/loss'] = yearly_closed_gain_loss_amount_moomoo


    # open df process
    # if not open_buy_df_previous.empty:
    yearly_trades_df_moomoo = pd.concat([yearly_trades_df_moomoo, open_buy_df_moomoo_previous], ignore_index=True)

    buy_filter_moomoo = (yearly_trades_df_moomoo['ActivityCode'] == 'BUY')
    sell_filter_moomoo = (yearly_trades_df_moomoo['ActivityCode'] == 'SELL')

    buy_df_moomoo = yearly_trades_df_moomoo[buy_filter_moomoo]
    sell_df_moomoo = yearly_trades_df_moomoo[sell_filter_moomoo]

    buy_ticker_grouped_moomoo = buy_df_moomoo.groupby('Symbol')['TradeQuantity'].sum()
    sell_ticker_grouped_moomoo = sell_df_moomoo.groupby('Symbol')['TradeQuantity'].sum()

    # Fill NaN values in sell_grouped with 0 (indicating no sales for those symbols)
    sell_ticker_grouped_moomoo = sell_ticker_grouped_moomoo.reindex(buy_ticker_grouped_moomoo.index, fill_value=0)
    # use '+' sign due to sell_groups contains negative quantity
    open_ticker_grouped_moomoo = buy_ticker_grouped_moomoo + sell_ticker_grouped_moomoo
    open_ticker_grouped_df_moomoo = open_ticker_grouped_moomoo[open_ticker_grouped_moomoo>0].reset_index()


    # Merging the open stock symbols with their details from the buy_df to create the open_stock_df
    open_buy_df_moomoo = pd.merge(
        open_ticker_grouped_df_moomoo
        ,buy_df_moomoo
        ,on='Symbol'
        ,how='left'
        )
    
    open_buy_df_moomoo = open_buy_df_moomoo.drop_duplicates(
        subset=[
            'Symbol'
            ,'TradeQuantity_x'
            ]
            ,keep='first'
            )



    # open_buy_df cleaning process
    selected_cols = [
        'ReportDate'
        ,'ActivityCode'
        ,'Buy/Sell'
        ,'TradeQuantity_x'
        ,'TradeGross'
        ,'Amount'
        ,'CurrencyPrimary'
        ,'AssetClass'
        ,'Symbol'
        ,'Year'
    ]

    open_buy_df_moomoo = open_buy_df_moomoo[selected_cols]

    open_buy_df_moomoo.rename(columns={
        'TradeQuantity_x': 'TradeQuantity'
        }
        ,inplace=True
        )


    buy_shares_qty_moomoo = abs(buy_ticker_grouped_moomoo.sum())
    sell_shares_qty_moomoo = abs(sell_ticker_grouped_moomoo.sum())
    open_shares_qty_moomoo = abs(buy_shares_qty_moomoo - sell_shares_qty_moomoo)


    # Save the current open_buy_df to be used in the next year's iteration
    open_buy_df_moomoo_previous = open_buy_df_moomoo.copy()

    print(i, open_shares_qty_moomoo, open_buy_df_moomoo['Symbol'].unique())
    # iterate the open stock list
    for symbol in open_buy_df_moomoo['Symbol'].unique():


        # STOCK SPLIT FACTOR section
        url = f'https://www.alphavantage.co/query?function=SPLITS&symbol={symbol}&apikey={alpha_vantage_api_key}'
        r = requests.get(url)
        data = r.json()

        for key, value in data.items():
            if key == 'data':
                if len(value) > 0:
                    stock_split_record_df = pd.DataFrame(value)
                    stock_split_record_df['split_factor'] = pd.to_numeric(stock_split_record_df['split_factor'], errors='coerce') # change split_factor series to numeric data
                    stock_split_record_df['effective_date'] = pd.to_datetime(stock_split_record_df['effective_date'])
                else:
                    # Use pandas index assign instead of direct assign value
                    stock_split_record_df = pd.DataFrame({
                    'split_factor': [1],
                    'effective_date': [datetime.today()]
                })


        # Daily quote section
        # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
        url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={alpha_vantage_api_key}&outputsize=full'
        r = requests.get(url)
        data = r.json()

        for key, value in data.items():
            if key == 'Time Series (Daily)':


                selected_cols = [
                    '4. close'
                ]

                Daily_stock_df = pd.DataFrame(value).transpose()[selected_cols] # tranpose the dataframe and sub select selected cols

                # Rename columns
                Daily_stock_df.rename(
                    columns={
                        '4. close': 'stock_price'
                        }
                    ,inplace=True
                    )
                
                Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].astype(str).apply(lambda x: float(x))
                Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].round(2)
                Daily_stock_df.index = pd.to_datetime(Daily_stock_df.index)


        for date_i in Daily_stock_df.index.date:
            for date_j in stock_split_record_df['effective_date'].dt.date:
                if date_i == date_j:

                    # stock price to divided the split factor
                    Daily_stock_df.loc[Daily_stock_df.index.date < date_j, 'stock_price'] /= (stock_split_record_df['split_factor'][stock_split_record_df['effective_date'].dt.date == date_j].values[0])


        # Group by year and get the last entry for each year, for the current year, will get the latest date quote
        latest_Daily_stock_df = Daily_stock_df.resample('Y').last()

        open_stock_price = latest_Daily_stock_df[latest_Daily_stock_df.index.year == i].values[0][0]
        open_stock_qty = open_buy_df_moomoo[open_buy_df_moomoo['Symbol'] == symbol]['TradeQuantity'].values.sum()


        open_stock_mkt_value = (open_stock_qty * open_stock_price).sum()
        open_stock_mkt_value_moomoo_list.append(open_stock_mkt_value)
        print(symbol, open_stock_qty, open_stock_price, open_stock_mkt_value)


    print(round(sum(open_stock_mkt_value_moomoo_list), 2))
    print()
    inv_yrly_base_moomoo.loc[inv_yrly_base_moomoo.index == i, 'Open_mkt_value'] = round(sum(open_stock_mkt_value_moomoo_list), 2)


    print()

2024 915.0 ['CE' 'KR' 'MSFT' 'SGOV' 'SOWG']
CE 42.0 69.21 2906.8199999999997
KR 50.0 61.15 3057.5
MSFT 7.0 421.5 2950.5
SGOV 100.0 100.32 10032.0
SOWG 716.0 30.584707646176913 21898.65067466267
40845.47


2025 222.0 ['ACN' 'CL' 'LPG' 'META' 'NVDA' 'V']
ACN 64.0 268.3 17171.2
CL 87.0 79.02 6874.74
LPG 10.0 24.34 243.4
META 14.0 660.09 9241.26
NVDA 22.0 186.5 4103.0
V 25.0 350.71 8767.75
46401.35


2026 141.0 ['ACN' 'INTU' 'META' 'NVDA' 'V']
ACN 64.0 178.71 11437.44
INTU 5.0 388.5 1942.5
META 14.0 611.91 8566.74
NVDA 22.0 199.57 4390.54
V 36.0 329.84 11874.24
38211.46




In [40]:
inv_yrly_base_moomoo

,Deposits,Withdrawals,Dividends,Closed_gain/loss,Open_mkt_value
Year,,,,,
2024,50802.884448,0.00,134.615655,-28180.721147,40845.47
2025,27500.000000,-12055.89,974.482836,-20459.374183,46401.35
2026,0.000000,-10000.00,0.000000,2984.143400,38211.46


# Data Source Gathering: open transactions

## Brokerage: Questrade

In [41]:
selected_cols = [
    'Transaction_Date'
    ,'Action'
    ,'Symbol'
    ,'Quantity'
    ,'Price'
    ,'Net_Amount'
    ,'Year'
]

trades_df_questrade = trades_df_questrade[selected_cols]

## Brokerage: MOOMOO

In [42]:
# Moomoo transaction history, the spreadsheet contains YTD transactions
# Moomoo transaction can only export 1 Yr from the web, so the full transactions will combine all years historical together
inv_activity_source_path_2024 = 'individual_activities_2024.csv'
inv_activity_source_path_2025 = 'individual_activities_2025.csv'
inv_activity_source_path_2026 = 'individual_activities_2026.csv'

inv_activity_df_moomoo_2024 = pd.read_csv(inv_activity_source_path_2024)
inv_activity_df_moomoo_2025 = pd.read_csv(inv_activity_source_path_2025)
inv_activity_df_moomoo_2026 = pd.read_csv(inv_activity_source_path_2026)

inv_activity_df_moomoo = pd.concat([
    inv_activity_df_moomoo_2024
    ,inv_activity_df_moomoo_2025
    ,inv_activity_df_moomoo_2026
])

In [43]:
selected_cols = [
    # DATE DIMENSION
    'ReportDate'

    ,'ActivityCode'
    ,'Buy/Sell'
    ,'TradeQuantity'
    ,'TradeGross'
    ,'Amount'

    ,'CurrencyPrimary'
    ,'AssetClass'
    ,'Symbol'
]

inv_activity_df_moomoo = inv_activity_df_moomoo[selected_cols]

# Assuming 'df' is the DataFrame
inv_activity_df_moomoo['ReportDate'] = pd.to_datetime(inv_activity_df_moomoo['ReportDate'], format='%Y%m%d')
# Assuming 'df' is the DataFrame
inv_activity_df_moomoo['Year'] = inv_activity_df_moomoo['ReportDate'].dt.year

In [44]:
filter_2 = (inv_activity_df_moomoo['ActivityCode'].isin([
    'FOREX'
    ,'BUY'
    ,'SELL'
    ]))

filter_3 = (inv_activity_df_moomoo['AssetClass'].isin([
    'CASH'
    ,'STK'
    ]))

filter_4 = (inv_activity_df_moomoo['CurrencyPrimary'].isin([
    'USD'
    ]))

trades_df_moomoo = inv_activity_df_moomoo[
    filter_2
    & filter_3
    & filter_4
    ]

In [45]:
trades_df_moomoo = trades_df_moomoo.rename(
    columns={
        'ReportDate': 'Transaction_Date'
        ,'ActivityCode': 'Action'
        ,'TradeQuantity': 'Quantity'
        ,'Amount': 'Net_Amount'
    }
)

# replace categorical value to prepare for merge
trades_df_moomoo['Action'] = trades_df_moomoo['Action'].replace(
    {
        'BUY': 'Buy'
        ,'SELL': 'Sell'
        }
    )

# add new columns to preare for merge
trades_df_moomoo['Price'] = trades_df_moomoo['TradeGross'] / trades_df_moomoo['Quantity']

selected_cols = [
    'Transaction_Date'
    ,'Action'
    ,'Symbol'
    ,'Quantity'
    ,'Price'
    ,'Net_Amount'
    ,'Year'
]

trades_df_moomoo = trades_df_moomoo[selected_cols]

In [46]:
trades_df_merge = pd.concat(
    [
        trades_df_questrade
        ,trades_df_moomoo
        ]
        , ignore_index=True
)

## Brokerage: RBC TFSA

In [47]:
# Create the dataframe
# Dividends, Closed_gain/loss, Open_mkt_value are converted to USD!!!
inv_yrly_base_rbc = pd.DataFrame({
    "Year": [2019, 2020, 2021, 2022, 2023, 2024],
    "Deposits": [0, 10000, 0, 0, 20000, 0],
    "Withdrawals": [0, 0, 0, 0, 0, -41083.87],
    "Dividends": [0, 0, 0, 0, 0, 0],
    "Closed_gain/loss": [0, -10000, -10000, -10000, -30000, 41083.87],
    "Open_mkt_value": [0, 10000, 10000, 10000, 30000, 0]
})

inv_yrly_base_rbc.set_index("Year", inplace=True)

inv_yrly_base_rbc = inv_yrly_base_rbc.merge(
    fx_CADUSD_yrly_df
    ,on='Year'
    ,how='left'
)

inv_yrly_base_rbc['Closed_gain/loss'] *= inv_yrly_base_rbc['CADUSD']
inv_yrly_base_rbc['Open_mkt_value'] *= inv_yrly_base_rbc['CADUSD']
inv_yrly_base_rbc.drop(columns=['CADUSD'], inplace=True)
inv_yrly_base_rbc.set_index("Year", inplace=True)

# Brokerage Consolidated

In [48]:
inv_yrly_base

,Deposits,Withdrawals,Dividends,Closed_gain/loss,Open_mkt_value
Year,,,,,
2019,6000.00,0.0,15.060000,-4668.270000,5217.78
2020,5825.00,0.0,9.420000,-3162.900000,5563.09
2021,10200.00,0.0,65.330000,1641.390000,11719.60
2022,2343.00,-14000.0,55.500000,118.220000,5628.60
2023,10000.00,0.0,69.320000,8470.100000,4160.00
2024,21843.89,0.0,183.210000,-9934.380000,19683.24
2025,140000.00,0.0,2328.995883,-120970.430280,144610.96
2026,-11.24,0.0,785.725988,7813.010466,142381.91


In [49]:
inv_yrly_base_moomoo

,Deposits,Withdrawals,Dividends,Closed_gain/loss,Open_mkt_value
Year,,,,,
2024,50802.884448,0.00,134.615655,-28180.721147,40845.47
2025,27500.000000,-12055.89,974.482836,-20459.374183,46401.35
2026,0.000000,-10000.00,0.000000,2984.143400,38211.46


In [50]:
inv_yrly_base_rbc

,Deposits,Withdrawals,Dividends,Closed_gain/loss,Open_mkt_value
Year,,,,,
2019,0,0.00,0,0.000000,0.000000
2020,10000,0.00,0,-7466.166667,7466.166667
2021,0,0.00,0,-7979.666667,7979.666667
2022,0,0.00,0,-7669.833333,7669.833333
2023,20000,0.00,0,-22253.250000,22253.250000
2024,0,-41083.87,0,29925.490908,0.000000


## Yearly investment return summary

In [51]:
inv_yrly_base = inv_yrly_base.reset_index(drop=False)
inv_yrly_base_moomoo = inv_yrly_base_moomoo.reset_index(drop=False)
# inv_yrly_base_rbc = inv_yrly_base_rbc.reset_index(drop=False)

inv_yrly_base_merge = pd.concat(
    [inv_yrly_base
     ,inv_yrly_base_moomoo
    #  ,inv_yrly_base_rbc
     ]).groupby(['Year']).sum().reset_index(drop=False)

# Fill NaN values with 0 so we can safely perform addition
inv_yrly_base_merge = inv_yrly_base_merge.fillna(0)

In [52]:
inv_yrly_base_merge

,Year,Deposits,Withdrawals,Dividends,Closed_gain/loss,Open_mkt_value
0,2019,6000.000000,0.00,15.060000,-4668.270000,5217.78
1,2020,5825.000000,0.00,9.420000,-3162.900000,5563.09
2,2021,10200.000000,0.00,65.330000,1641.390000,11719.60
3,2022,2343.000000,-14000.00,55.500000,118.220000,5628.60
4,2023,10000.000000,0.00,69.320000,8470.100000,4160.00
5,2024,72646.774448,0.00,317.825655,-38115.101147,60528.71
6,2025,167500.000000,-12055.89,3303.478719,-141429.804463,191012.31
7,2026,-11.240000,-10000.00,785.725988,10797.153866,180593.37


In [53]:
# FOREX section
start_yr = inv_yrly_base_merge['Year'].min()  # Earliest year in 'Year'
end_yr = inv_yrly_base_merge['Year'].max()    # Latest year in 'Year'


yearly_exchange_rates = fx_CADUSD_yrly_df[
    (fx_CADUSD_yrly_df['Year'] >= start_yr) & (fx_CADUSD_yrly_df['Year'] <= end_yr)
]

CADUSD_mapping = dict(zip(yearly_exchange_rates['Year'], yearly_exchange_rates['CADUSD']))
inv_yrly_base_merge['CADUSD_forx'] = inv_yrly_base_merge['Year'].map(CADUSD_mapping)

In [54]:
inv_yrly_base_merge

,Year,Deposits,Withdrawals,Dividends,Closed_gain/loss,Open_mkt_value,CADUSD_forx
0,2019,6000.000000,0.00,15.060000,-4668.270000,5217.78,0.755675
1,2020,5825.000000,0.00,9.420000,-3162.900000,5563.09,0.746617
2,2021,10200.000000,0.00,65.330000,1641.390000,11719.60,0.797967
3,2022,2343.000000,-14000.00,55.500000,118.220000,5628.60,0.766983
4,2023,10000.000000,0.00,69.320000,8470.100000,4160.00,0.741775
5,2024,72646.774448,0.00,317.825655,-38115.101147,60528.71,0.728400
6,2025,167500.000000,-12055.89,3303.478719,-141429.804463,191012.31,0.715667
7,2026,-11.240000,-10000.00,785.725988,10797.153866,180593.37,0.737100


In [55]:
inv_yrly_base_merge['Deposits'] = round(inv_yrly_base_merge['Deposits'] * inv_yrly_base_merge['CADUSD_forx'], 2)
inv_yrly_base_merge['Withdrawals'] = round(inv_yrly_base_merge['Withdrawals'] * inv_yrly_base_merge['CADUSD_forx'], 2)

# Custom columns
inv_yrly_base_merge['Activity_Yr'] = inv_yrly_base_merge['Year']
inv_yrly_base_merge['Principal'] = inv_yrly_base_merge['Deposits'] + inv_yrly_base_merge['Withdrawals']
inv_yrly_base_merge['Principal_after_div'] = inv_yrly_base_merge['Principal'] + inv_yrly_base_merge['Dividends']


inv_yrly_base_merge['Running_Principal'] = inv_yrly_base_merge['Principal'].cumsum()
inv_yrly_base_merge['Running_Dividends'] = inv_yrly_base_merge['Dividends'].cumsum()
inv_yrly_base_merge['Running_Closed_gain/loss'] = inv_yrly_base_merge['Closed_gain/loss'].cumsum()
inv_yrly_base_merge['Running_Principal_after_div'] = inv_yrly_base_merge['Principal_after_div'].cumsum() # The real principal after commisions and dividends

inv_yrly_base_merge['Year_end_capital_value'] = inv_yrly_base_merge['Running_Principal_after_div'] \
                                        + inv_yrly_base_merge['Running_Closed_gain/loss'] \
                                        + inv_yrly_base_merge['Open_mkt_value'] \


for i in inv_yrly_base_merge['Activity_Yr'].unique():

        previous_year_end_capital_value = inv_yrly_base_merge[inv_yrly_base_merge['Activity_Yr'] == (i-1)]['Year_end_capital_value']

        if previous_year_end_capital_value.empty:
            previous_year_end_capital_value = 0

        else:
            previous_year_end_capital_value = inv_yrly_base_merge[inv_yrly_base_merge['Activity_Yr'] == (i-1)]['Year_end_capital_value'].values[0]

        current_principal = inv_yrly_base_merge[inv_yrly_base_merge['Activity_Yr'] == i]['Principal'].values[0]
        inv_yrly_base_merge.loc[inv_yrly_base_merge['Activity_Yr'] == i, 'Year_start_capital_value'] = previous_year_end_capital_value + current_principal





# Ratio Metrics
inv_yrly_base_merge['Ratio_Dividends_Yield_%'] = (inv_yrly_base_merge['Dividends'] / inv_yrly_base_merge['Running_Principal_after_div']) * 100
inv_yrly_base_merge['Ratio_Yearly_Return_%'] = round(((inv_yrly_base_merge['Year_end_capital_value'] - inv_yrly_base_merge['Year_start_capital_value']) / inv_yrly_base_merge['Year_start_capital_value']) * 100, 2)

####################
# CAGR calculation #
# NOT IN USE       #
####################
# CAGR_ending_value_fund = inv_yrly_base_merge['Year_end_capital_value'].iloc[-1] - inv_yrly_base_merge['Running_Principal'].iloc[-1] # exclude the total principal from the year end fund value
# CAGR_starting_value_fund = inv_yrly_base_merge['Year_end_capital_value'].iloc[0] - inv_yrly_base_merge['Running_Principal'].iloc[0]
CAGR_ending_value_sp500 = sp500_hist_annual[sp500_hist_annual['Year'] == datetime.today().year]['stock_price'].values.round(2)[0] # at today's closing point
CAGR_starting_value_sp500 = sp500_hist_annual[sp500_hist_annual['Year'] == 2018]['stock_price'].values.round(2)[0]  # at 2018 closing point equal to 2019 opening
CAGR_years = inv_yrly_base_merge.shape[0]
# NOTE: for below CAGR formula, it cannot intake negatives ending value, if the ending value is negatives, it will return NaN.
# if it returns NaN, please use excel spreadsheet to calculate the CAGR_ending_value_fund, because excel can handle the negatives ending value

#########################################
# TWR(time weighted return) calculation #
# IN PRODUCTION                         #
#########################################
TWR_list = []
for r in inv_yrly_base_merge['Ratio_Yearly_Return_%'].values:
     TWR_list.append((r / 100) + 1)

inv_yrly_base_merge['CAGR_%'] = (math.prod(TWR_list) - 1) * 100  
inv_yrly_base_merge['sp500_CAGR_%'] =  ((CAGR_ending_value_sp500 / CAGR_starting_value_sp500) ** (1 / CAGR_years) - 1) * 100


# drop yr as index
inv_yrly_base_merge.reset_index(drop=True,inplace=True)

# join on sp500
inv_yrly_merged = inv_yrly_base_merge.merge (
    sp500_hist_annual
    ,how='left'
    ,left_on='Activity_Yr'
    ,right_on='Year'
)


inv_yrly_merged.rename(
    columns={
    'Annual Return': 'sp500_without_div_%'
    }
    ,inplace=True
    )

In [56]:
inv_yrly_base_merge

,Year,Deposits,Withdrawals,Dividends,Closed_gain/loss,Open_mkt_value,CADUSD_forx,Activity_Yr,Principal,Principal_after_div,Running_Principal,Running_Dividends,Running_Closed_gain/loss,Running_Principal_after_div,Year_end_capital_value,Year_start_capital_value,Ratio_Dividends_Yield_%,Ratio_Yearly_Return_%,CAGR_%,sp500_CAGR_%
0,2019,4534.05,0.00,15.060000,-4668.270000,5217.78,0.755675,2019,4534.05,4549.110000,4534.05,15.060000,-4668.270000,4549.110000,5098.620000,4534.050000,0.331054,12.45,21.721966,14.114362
1,2020,4349.04,0.00,9.420000,-3162.900000,5563.09,0.746617,2020,4349.04,4358.460000,8883.09,24.480000,-7831.170000,8907.570000,6639.490000,9447.660000,0.105753,-29.72,21.721966,14.114362
2,2021,8139.26,0.00,65.330000,1641.390000,11719.60,0.797967,2021,8139.26,8204.590000,17022.35,89.810000,-6189.780000,17112.160000,22641.980000,14778.750000,0.381775,53.21,21.721966,14.114362
3,2022,1797.04,-10737.77,55.500000,118.220000,5628.60,0.766983,2022,-8940.73,-8885.230000,8081.62,145.310000,-6071.560000,8226.930000,7783.970000,13701.250000,0.674614,-43.19,21.721966,14.114362
4,2023,7417.75,0.00,69.320000,8470.100000,4160.00,0.741775,2023,7417.75,7487.070000,15499.37,214.630000,2398.540000,15714.000000,22272.540000,15201.720000,0.441135,46.51,21.721966,14.114362
5,2024,52915.91,0.00,317.825655,-38115.101147,60528.71,0.728400,2024,52915.91,53233.735655,68415.28,532.455655,-35716.561147,68947.735655,93759.884508,75188.450000,0.460966,24.70,21.721966,14.114362
6,2025,119874.17,-8628.00,3303.478719,-141429.804463,191012.31,0.715667,2025,111246.17,114549.648719,179661.45,3835.934374,-177146.365610,183497.384374,197363.328764,205006.054508,1.800287,-3.73,21.721966,14.114362
7,2026,-8.29,-7371.00,785.725988,10797.153866,180593.37,0.737100,2026,-7379.29,-6593.564012,172282.16,4621.660362,-166349.211744,176903.820362,191147.978618,189984.038764,0.444154,0.61,21.721966,14.114362


In [57]:
selected_cols = [
    'Activity_Yr'
    ,'Ratio_Yearly_Return_%'
    ,'sp500_without_div_%'
    ,'CAGR_%'
    ,'sp500_CAGR_%'
]

# Specify the columns you want to round
columns_to_round = [
    'Ratio_Yearly_Return_%'
    ,'sp500_without_div_%'
    ,'CAGR_%'
    ,'sp500_CAGR_%'
    ]

# Round the selected columns to 2 decimals
inv_yrly_merged[columns_to_round] = inv_yrly_merged[columns_to_round].round(2)

inv_yrly_merged_consolidate = inv_yrly_merged[selected_cols]

In [58]:
inv_yrly_merged_consolidate

,Activity_Yr,Ratio_Yearly_Return_%,sp500_without_div_%,CAGR_%,sp500_CAGR_%
0,2019,12.45,28.79,21.72,14.11
1,2020,-29.72,16.16,21.72,14.11
2,2021,53.21,27.04,21.72,14.11
3,2022,-43.19,-19.48,21.72,14.11
4,2023,46.51,24.29,21.72,14.11
5,2024,24.70,23.30,21.72,14.11
6,2025,-3.73,16.35,21.72,14.11
7,2026,0.61,5.39,21.72,14.11


## Stock open position holding cost calculator

In [59]:
# Initialize open_buy_df_previous as an empty DataFrame
open_buy_df_previous = pd.DataFrame()
year_end_open_stock_dict = {} # this dict contains all the open stock positions of each year end, current year will be the current open position

# sort the year from the smallest to the largest to match open stock sequence
for i in sorted(trades_df_merge['Year'].unique()):

    # store the key, value pair in the year_end_open_stock_dict, and build nested dictionary for each year data
    # 1st layer dictionary
    # year is the outer key
    year_end_open_stock_dict[i] = {
        'Symbol': []
        ,'Quantity': []
    }

    open_stock_mkt_value_list = []
    yearly_trades_df = trades_df_merge[trades_df_merge['Year'] == i]

    # if not open_buy_df_previous.empty:
    yearly_trades_df = pd.concat([yearly_trades_df, open_buy_df_previous], ignore_index=True)
    yearly_closed_gain_loss_amount = yearly_trades_df['Net_Amount'].sum()

    # first assign net_amount to the inv_yrly_base
    inv_yrly_base.loc[inv_yrly_base.index == i, 'Closed_gain/loss'] = yearly_closed_gain_loss_amount

    buy_filter = (yearly_trades_df['Action'] == 'Buy')
    sell_filter = (yearly_trades_df['Action'] == 'Sell')


    buy_df = yearly_trades_df[buy_filter]
    sell_df = yearly_trades_df[sell_filter]

    buy_ticker_grouped = buy_df.groupby('Symbol')['Quantity'].sum()
    sell_ticker_grouped = sell_df.groupby('Symbol')['Quantity'].sum()

    # Fill NaN values in sell_grouped with 0 (indicating no sales for those symbols)
    sell_ticker_grouped = sell_ticker_grouped.reindex(buy_ticker_grouped.index, fill_value=0)
    # use '+' sign due to sell_groups contains negative quantity
    open_ticker_grouped = buy_ticker_grouped + sell_ticker_grouped
    open_ticker_grouped_df = open_ticker_grouped[open_ticker_grouped>0].reset_index()


    # Merging the open stock symbols with their details from the buy_df to create the open_stock_df
    open_buy_df = pd.merge(
        open_ticker_grouped_df
        ,buy_df
        ,on='Symbol'
        ,how='left'
        )
    
    open_buy_df = open_buy_df.drop_duplicates(
        subset=[
            'Symbol'
            ,'Quantity_x'
            ]
            ,keep='first'
            )


    # open_buy_df cleaning process
    selected_cols = [
    'Transaction_Date'
    ,'Action'
    ,'Symbol'
    ,'Quantity_x'
    ,'Price'
    ,'Net_Amount'
    ,'Year'
    ]

    open_buy_df = open_buy_df[selected_cols]

    open_buy_df.rename(columns={
        'Quantity_x': 'Quantity'
        }
        ,inplace=True
        )


    buy_shares_qty = abs(buy_ticker_grouped.sum())
    sell_shares_qty = abs(sell_ticker_grouped.sum())
    open_shares_qty = abs(buy_shares_qty - sell_shares_qty)


    # Save the current open_buy_df to be used in the next year's iteration
    open_buy_df_previous = open_buy_df.copy()

    print(i, open_shares_qty, open_buy_df['Symbol'].unique())
    # iterate the open stock list
    for symbol in open_buy_df['Symbol'].unique():


        # STOCK SPLIT FACTOR section
        url = f'https://www.alphavantage.co/query?function=SPLITS&symbol={symbol}&apikey={alpha_vantage_api_key}'
        r = requests.get(url)
        data = r.json()

        for key, value in data.items():
            if key == 'data':
                if len(value) > 0:
                    stock_split_record_df = pd.DataFrame(value)
                    stock_split_record_df['split_factor'] = pd.to_numeric(stock_split_record_df['split_factor'], errors='coerce') # change split_factor series to numeric data
                    stock_split_record_df['effective_date'] = pd.to_datetime(stock_split_record_df['effective_date'])
                else:
                    # Use pandas index assign instead of direct assign value
                    stock_split_record_df = pd.DataFrame({
                    'split_factor': [1],
                    'effective_date': [datetime.today()]
                })


        # Daily quote section
        # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
        url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={alpha_vantage_api_key}&outputsize=full'
        r = requests.get(url)
        data = r.json()

        for key, value in data.items():
            if key == 'Time Series (Daily)':


                selected_cols = [
                    '4. close'
                ]

                Daily_stock_df = pd.DataFrame(value).transpose()[selected_cols] # tranpose the dataframe and sub select selected cols

                # Rename columns
                Daily_stock_df.rename(
                    columns={
                        '4. close': 'stock_price'
                        }
                    ,inplace=True
                    )
                
                Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].astype(str).apply(lambda x: float(x))
                Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].round(2)
                Daily_stock_df.index = pd.to_datetime(Daily_stock_df.index)


        for date_i in Daily_stock_df.index.date:
            for date_j in stock_split_record_df['effective_date'].dt.date:
                if date_i == date_j:

                    # stock price to divided the split factor
                    Daily_stock_df.loc[Daily_stock_df.index.date < date_j, 'stock_price'] /= (stock_split_record_df['split_factor'][stock_split_record_df['effective_date'].dt.date == date_j].values[0])


        # Group by year and get the last entry for each year, for the current year, will get the latest date quote
        latest_Daily_stock_df = Daily_stock_df.resample('Y').last()

        open_stock_price = latest_Daily_stock_df[latest_Daily_stock_df.index.year == i].values[0][0]
        open_stock_qty = open_buy_df[open_buy_df['Symbol'] == symbol]['Quantity'].values.sum()


        open_stock_mkt_value = (open_stock_qty * open_stock_price).sum()
        open_stock_mkt_value_list.append(open_stock_mkt_value)
        print(symbol, open_stock_qty, open_stock_price, open_stock_mkt_value)


        # store the key, value pair in the year_end_open_stock_dict, and build nested dictionary for each year data
        # 2nd layer dictionary
        # symbol is the 1st inner key, which using a list as value to store
        # quantity is the 2nd inner key, which using a list as value to store
        year_end_open_stock_dict[i]['Symbol'].append(symbol)
        year_end_open_stock_dict[i]['Quantity'].append(open_stock_qty)

    print(round(sum(open_stock_mkt_value_list), 2))
    print()




2019 41.0 ['BABA' 'DDOG' 'ORCL']
BABA 20.0 212.1 4242.0
DDOG 9.0 37.78 340.02
ORCL 12.0 52.98 635.76
5217.78

2020 72.0 ['AAL' 'AMZN' 'DDOG']
AAL 20.0 15.77 315.4
AMZN 2.0 162.8465 325.693
DDOG 50.0 98.44 4922.0
5563.09

2021 50.0 ['FDX' 'META' 'TSM']
FDX 10.0 258.64 2586.3999999999996
META 20.0 336.35 6727.0
TSM 20.0 120.31 2406.2
11719.6

2022 50.0 ['FDX' 'META' 'TSM']
FDX 10.0 173.2 1732.0
META 20.0 120.34 2406.8
TSM 20.0 74.49 1489.8
5628.6

2023 40.0 ['TSM']
TSM 40.0 104.0 4160.0
4160.0

2024 1024.0 ['CE' 'GOOG' 'KR' 'MSFT' 'SGOV' 'SOWG' 'TGT' 'TSM']
CE 42.0 69.21 2906.8199999999997
GOOG 67.0 190.44 12759.48
KR 50.0 61.15 3057.5
MSFT 7.0 421.5 2950.5
SGOV 100.0 100.32 10032.0
SOWG 716.0 30.584707646176913 21898.65067466267
TGT 22.0 135.18 2973.96
TSM 20.0 197.49 3949.8
60528.71

2025 3419.0 ['ACN' 'CL' 'CMR.TO' 'GOOG' 'LPG' 'META' 'NVDA' 'SGOV' 'TGT' 'TSM' 'V']
ACN 71.0 268.3 19049.3
CL 199.0 79.02 15724.98
CMR.TO 2796.0 50.01 139827.96
GOOG 10.0 313.8 3138.0
LPG 10.0 24.34 243.4


In [60]:
current_open_holding_df = pd.DataFrame()
current_yr_open_stock_list = year_end_open_stock_dict[datetime.today().year]['Symbol']

for symbol in set(current_yr_open_stock_list):
        current_open_holding_df = pd.concat(
            [
                current_open_holding_df
                , trades_df_merge[trades_df_merge['Symbol'] == symbol]
                ]
                ).drop_duplicates()
        
current_open_holding_df = current_open_holding_df.sort_values(['Transaction_Date'], ascending=True)

In [61]:
current_open_holding_cost_dict = {}

for symbol in current_open_holding_df['Symbol'].unique():
    
    open_symbol_holding_df = current_open_holding_df[current_open_holding_df['Symbol'] == symbol]

    # open_symbol_holding_df['Total_Capital'] = Total_capital # 总资金, 不包括unrealized capital gain
    open_symbol_holding_df['Rolling_Capital'] = open_symbol_holding_df['Net_Amount'].cumsum() # 个股每笔交易后的投入
    open_symbol_holding_df['Rolling_Quantity'] = open_symbol_holding_df['Quantity'].cumsum() # 个股每笔交易后的股票数
    open_symbol_holding_df['Rolling_Price_Per_Share'] = open_symbol_holding_df['Rolling_Capital'] / open_symbol_holding_df['Rolling_Quantity'] # 个股每笔交易后的持仓成本
    # open_symbol_holding_df['Rolling_Capital_Percentage'] = abs(open_symbol_holding_df['Rolling_Capital'] / open_symbol_holding_df['Total_Capital']).round(2) * 100 # 个股每笔交易后, 仓位所占总投入的百分比

    # update the dataframe with columns which are needed
    selected_cols = [
        'Transaction_Date'
        ,'Symbol'
        ,'Action'
        ,'Quantity'
        ,'Rolling_Quantity'
        ,'Price'
        ,'Rolling_Capital'
        ,'Rolling_Price_Per_Share'
        # ,'Rolling_Capital_Percentage'
        # ,'Total_Capital'
    ]
    open_symbol_holding_df = open_symbol_holding_df[selected_cols]

    # appending each ticker's ticker name as key, and dataframe as value into the dictionary
    current_open_holding_cost_dict[symbol] = open_symbol_holding_df


In [62]:
consolidated_holding_cost_df = pd.DataFrame()
for value in current_open_holding_cost_dict.values():
    consolidated_holding_cost_df = pd.concat([consolidated_holding_cost_df, value])



# Manually make date difference if same transaction date happened in the same ticker, to avoid miscalculation
# Group by Symbol and Transaction_Date
dupe_groups = consolidated_holding_cost_df.groupby(['Symbol', 'Transaction_Date']).cumcount()

# Apply offset where there are duplicates, unit'd' means one day difference
consolidated_holding_cost_df['Transaction_Date'] = consolidated_holding_cost_df['Transaction_Date'] + pd.to_timedelta(dupe_groups, unit='d')

## Stock open position contribution mix

In [63]:
# Get the latest transaction for each stock based on 'Transaction_Date'
consolidated_holding_cost_df_latest = consolidated_holding_cost_df.loc[consolidated_holding_cost_df.groupby('Symbol')['Transaction_Date'].idxmax()]

# Reset index if needed
consolidated_holding_cost_df_latest = consolidated_holding_cost_df_latest.reset_index(drop=True)
# Create empty position_% column
consolidated_holding_cost_df_latest['position_%'] = np.empty

In [64]:
for symbol in consolidated_holding_cost_df_latest['Symbol'].unique():

    # STOCK SPLIT FACTOR section
    url = f'https://www.alphavantage.co/query?function=SPLITS&symbol={symbol}&apikey={alpha_vantage_api_key}'
    r = requests.get(url)
    data = r.json()

    for key, value in data.items():
        if key == 'data':
            if len(value) > 0:
                stock_split_record_df = pd.DataFrame(value)
                stock_split_record_df['split_factor'] = pd.to_numeric(stock_split_record_df['split_factor'], errors='coerce') # change split_factor series to numeric data
                stock_split_record_df['effective_date'] = pd.to_datetime(stock_split_record_df['effective_date'])
            else:
                # Use pandas index assign instead of direct assign value
                stock_split_record_df = pd.DataFrame({
                'split_factor': [1],
                'effective_date': [datetime.today()]
            })


    # Daily quote section
    # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
    url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={alpha_vantage_api_key}&outputsize=full'
    r = requests.get(url)
    data = r.json()

    for key, value in data.items():
        if key == 'Time Series (Daily)':


            selected_cols = [
                '4. close'
            ]

            Daily_stock_df = pd.DataFrame(value).transpose()[selected_cols] # tranpose the dataframe and sub select selected cols

            # Rename columns
            Daily_stock_df.rename(
                columns={
                    '4. close': 'stock_price'
                    }
                ,inplace=True
                )
            
            Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].astype(str).apply(lambda x: float(x))
            Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].round(2)
            Daily_stock_df.index = pd.to_datetime(Daily_stock_df.index)


    for date_i in Daily_stock_df.index.date:
        for date_j in stock_split_record_df['effective_date'].dt.date:
            if date_i == date_j:

                # stock price to divided the split factor
                Daily_stock_df.loc[Daily_stock_df.index.date < date_j, 'stock_price'] /= (stock_split_record_df['split_factor'][stock_split_record_df['effective_date'].dt.date == date_j].values[0])


    latest_date_stock_price = Daily_stock_df['stock_price'].iloc[0]
    consolidated_holding_cost_df_latest.loc[consolidated_holding_cost_df_latest['Symbol'] == symbol, 'latest_stock_price'] = latest_date_stock_price



    # position_% calculation
    rolling_price_per_share = consolidated_holding_cost_df_latest[consolidated_holding_cost_df_latest['Symbol'] == symbol]['Rolling_Price_Per_Share'].values
    rolling_qty = consolidated_holding_cost_df_latest[consolidated_holding_cost_df_latest['Symbol'] == symbol]['Rolling_Quantity'].values
    rolling_capital = consolidated_holding_cost_df_latest[consolidated_holding_cost_df_latest['Symbol'] == symbol]['Rolling_Capital'].values

    if rolling_capital < 0:
        holding_cost = abs(rolling_capital)

    if rolling_capital >= 0:
        # need to revise logic
        holding_cost = 1

    # change 'Running_Principal_after_div' to 'Year_end_capital_value' to calculate the position_%
    consolidated_holding_cost_df_latest.loc[(consolidated_holding_cost_df_latest['Symbol'] == symbol), 'position_%'] = ((holding_cost / (inv_yrly_merged['Year_end_capital_value'].values[-1])) * 100).round(2)
    consolidated_holding_cost_df_latest.loc[(consolidated_holding_cost_df_latest['Symbol'] == symbol), 'Year_end_capital_value'] = inv_yrly_merged['Year_end_capital_value'].values[-1]


In [65]:
consolidated_holding_cost_df_latest

,Transaction_Date,Symbol,Action,Quantity,Rolling_Quantity,Price,Rolling_Capital,Rolling_Price_Per_Share,position_%,latest_stock_price,Year_end_capital_value
0,2026-04-17,ACN,Buy,20.0,119.0,196.447000,-27355.091737,-229.874720,14.31,178.71,191147.978618
1,2026-02-05,CMR.TO,Sell,-98.0,1398.0,36.869742,-48641.579814,-34.793691,25.45,50.02,191147.978618
2,2025-09-12,GOOG,Sell,-57.0,10.0,239.780000,7293.286100,729.328610,0.0,381.94,191147.978618
3,2026-04-17,INTU,Buy,15.0,46.0,400.500000,-17655.378000,-383.812565,9.24,388.50,191147.978618
4,2025-11-10,META,Buy,14.0,14.0,-633.100000,-8851.900000,-632.278571,4.63,611.91,191147.978618
5,2026-04-25,MSFT,Buy,9.0,63.0,420.539000,-23880.338495,-379.052992,12.49,407.78,191147.978618
6,2025-11-26,NVDA,Buy,28.0,50.0,176.250000,-9405.840000,-188.116800,4.92,199.57,191147.978618
7,2026-02-25,SKWD,Buy,24.0,488.0,43.613600,-22492.290000,-46.090758,11.77,45.45,191147.978618
8,2024-07-17,TGT,Buy,22.0,22.0,156.049900,-3438.050000,-156.275000,1.8,129.75,191147.978618
9,2025-08-19,TSM,Sell,-15.0,5.0,241.038100,5595.459300,1119.091860,0.0,396.06,191147.978618


In [66]:
selected_cols = [
    'Transaction_Date'
    ,'Symbol'
    ,'Rolling_Quantity'
    ,'Rolling_Capital'
    ,'Rolling_Price_Per_Share'
    ,'position_%'
    # ,'Running_Principal_after_div'
    ,'Year_end_capital_value'
    ,'latest_stock_price'
]


consolidated_holding_cost_df_latest = consolidated_holding_cost_df_latest[selected_cols]

## Next qtr earning date

In [67]:
# # forecast 1 qtr earnings from alpha vantage API
# for i in earning_calendar: comment out the for loop in case of future usage, i can be the parameter of {}month
CSV_URL = f'https://www.alphavantage.co/query?function=EARNINGS_CALENDAR&symbol={symbol}&horizon=12month&apikey={alpha_vantage_api_key}'
with requests.Session() as s:
    download = s.get(CSV_URL)
    decoded_content = download.content.decode('utf-8')
    cr = csv.reader(decoded_content.splitlines(), delimiter=',')
    my_list = list(cr)

    forecast_earanings_df = pd.DataFrame(
        columns=my_list[0]
        ,data=my_list[1::]
        )

In [68]:
consolidated_holding_cost_df_latest['Next_qtr_date'] = None

for symbol in consolidated_holding_cost_df_latest['Symbol']:
    # Next earning calendar
    # for i in earning_calendar: comment out the for loop in case of future usage, i can be the parameter of {}month
    CSV_URL = f'https://www.alphavantage.co/query?function=EARNINGS_CALENDAR&symbol={symbol}&horizon=12month&apikey={alpha_vantage_api_key}'
    with requests.Session() as s:
        download = s.get(CSV_URL)
        decoded_content = download.content.decode('utf-8')
        cr = csv.reader(decoded_content.splitlines(), delimiter=',')
        my_list = list(cr)

        forecast_earanings_df = pd.DataFrame(
            columns=my_list[0]
            ,data=my_list[1::]
            )
        
        # Ensure reportDate exists in the dataframe
        if len(forecast_earanings_df['reportDate'].head(1).values) > 0:
            next_qtr_date = forecast_earanings_df['reportDate'].head(1).values[0]
            consolidated_holding_cost_df_latest.loc[consolidated_holding_cost_df_latest['Symbol'] == symbol, 'Next_qtr_date'] = next_qtr_date

        else:
            consolidated_holding_cost_df_latest.loc[consolidated_holding_cost_df_latest['Symbol'] == symbol, 'Next_qtr_date'] = None


In [69]:
CSV_URL = f'https://www.alphavantage.co/query?function=EARNINGS_CALENDAR&symbol=SOWG&horizon=12month&apikey={alpha_vantage_api_key}'
with requests.Session() as s:
    download = s.get(CSV_URL)
    decoded_content = download.content.decode('utf-8')
    cr = csv.reader(decoded_content.splitlines(), delimiter=',')
    my_list = list(cr)

    forecast_earanings_df = pd.DataFrame(
        columns=my_list[0]
        ,data=my_list[1::]
        )

# Display

In [70]:
inv_yrly_merged_consolidate

,Activity_Yr,Ratio_Yearly_Return_%,sp500_without_div_%,CAGR_%,sp500_CAGR_%
0,2019,12.45,28.79,21.72,14.11
1,2020,-29.72,16.16,21.72,14.11
2,2021,53.21,27.04,21.72,14.11
3,2022,-43.19,-19.48,21.72,14.11
4,2023,46.51,24.29,21.72,14.11
5,2024,24.70,23.30,21.72,14.11
6,2025,-3.73,16.35,21.72,14.11
7,2026,0.61,5.39,21.72,14.11


In [71]:
# latest unique stock holding
consolidated_holding_cost_df_latest.sort_values(by='position_%', ascending=False)

,Transaction_Date,Symbol,Rolling_Quantity,Rolling_Capital,Rolling_Price_Per_Share,position_%,Year_end_capital_value,latest_stock_price,Next_qtr_date
1,2026-02-05,CMR.TO,1398.0,-48641.579814,-34.793691,25.45,191147.978618,50.02,None
0,2026-04-17,ACN,119.0,-27355.091737,-229.874720,14.31,191147.978618,178.71,2026-06-18
5,2026-04-25,MSFT,63.0,-23880.338495,-379.052992,12.49,191147.978618,407.78,f
7,2026-02-25,SKWD,488.0,-22492.290000,-46.090758,11.77,191147.978618,45.45,f
3,2026-04-17,INTU,46.0,-17655.378000,-383.812565,9.24,191147.978618,388.50,2026-05-28
10,2026-04-15,V,45.0,-15252.905100,-338.953447,7.98,191147.978618,329.84,2026-07-28
6,2025-11-26,NVDA,50.0,-9405.840000,-188.116800,4.92,191147.978618,199.57,f
4,2025-11-10,META,14.0,-8851.900000,-632.278571,4.63,191147.978618,611.91,f
8,2024-07-17,TGT,22.0,-3438.050000,-156.275000,1.8,191147.978618,129.75,2026-05-20
2,2025-09-12,GOOG,10.0,7293.286100,729.328610,0.0,191147.978618,381.94,None


In [72]:
# latest unique stock holding
port_holding_pct = consolidated_holding_cost_df_latest.sort_values(by='position_%', ascending=False)['position_%'].sum()
port_cash_pct = 100 - port_holding_pct

# Print with two decimal places
print(f"Portfolio holding position: {port_holding_pct:.2f}")
print(f"Cash posistion: {port_cash_pct:.2f}")

Portfolio holding position: 92.59
Cash posistion: 7.41
